<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Module_11_own_coding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [81]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [82]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [83]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# train test split


In [84]:
X = df.drop('diagnosis', axis = 1)
y = df['diagnosis']

In [85]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Scaling

In [86]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Label encoding

In [87]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

# Numpy arrays to PyTorch tensors

In [88]:
X_train_tensor = torch.tensor(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

# Defining the model

In [89]:
class SimpleNN():
  def __init__(self, X):
    self.weights = torch.rand(X.shape[1], 1, dtype = torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype = torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_func(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # calculate loss
    loss = -torch.mean(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred))
    return loss

# Important Parameters

In [90]:
learning_rate = 0.1
epochs = 25

# Training Pipeline

In [92]:
# create model
model = SimpleNN(X_train_tensor)

# define loop
for epoch in range(epochs):
  # forward pass
  y_pred = model.forward(X_train_tensor)

  # loss calculate
  loss = model.loss_func(y_pred, y_train_tensor)

  # backward pass
  loss.backward()

  # parameters update
  with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad

  # zero gradient
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 3.533205546219525
Epoch: 2, Loss: 3.39435448012133
Epoch: 3, Loss: 3.2528655396037287
Epoch: 4, Loss: 3.1088067425631
Epoch: 5, Loss: 2.957712974960137
Epoch: 6, Loss: 2.8027305619310923
Epoch: 7, Loss: 2.635496430787087
Epoch: 8, Loss: 2.46324772283825
Epoch: 9, Loss: 2.2948227863050485
Epoch: 10, Loss: 2.1257896330525057
Epoch: 11, Loss: 1.950881974782554
Epoch: 12, Loss: 1.7762090550117182
Epoch: 13, Loss: 1.6079596177274578
Epoch: 14, Loss: 1.4505968063436354
Epoch: 15, Loss: 1.305237209398534
Epoch: 16, Loss: 1.177894271238968
Epoch: 17, Loss: 1.071069686314829
Epoch: 18, Loss: 0.986190319711269
Epoch: 19, Loss: 0.9224926856613485
Epoch: 20, Loss: 0.8765590186177429
Epoch: 21, Loss: 0.8439194954846728
Epoch: 22, Loss: 0.8206584683132703
Epoch: 23, Loss: 0.8037901061820484
Epoch: 24, Loss: 0.7911802433771361
Epoch: 25, Loss: 0.7813798000024902


In [93]:
model.bias

tensor([-0.1466], dtype=torch.float64, requires_grad=True)

# Evaluation

In [96]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.6206524968147278
